# Lab 3: Advanced Iceberg Features - Solution

This notebook contains the complete solution for Lab 3. Use this to verify your implementation or if you get stuck.

## Step 1: Advanced Partitioning Strategies - Partition Evolution

In [ ]:
// Start with a simple partitioned table
spark.sql("""
  CREATE TABLE iceberg.default.sales (
    sale_id INT,
    product_id INT,
    sale_date DATE,
    amount DECIMAL(10,2),
    region STRING,
    salesperson STRING
  ) USING iceberg
  PARTITIONED BY (region)
  TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet'
  )
""")

In [ ]:
// Insert initial data
spark.sql("""
  INSERT INTO iceberg.default.sales VALUES
    (1, 101, DATE '2024-01-01', 1000.50, 'east', 'alice'),
    (2, 102, DATE '2024-01-01', 750.25, 'west', 'bob'),
    (3, 103, DATE '2024-01-02', 500.75, 'east', 'alice'),
    (4, 104, DATE '2024-01-02', 1200.00, 'west', 'charlie')
""")

In [ ]:
// Add second partition field (partition evolution)
spark.sql("""
  ALTER TABLE iceberg.default.sales
  ADD PARTITION FIELD days(sale_date)
""")

In [ ]:
// Insert data with new partition structure
spark.sql("""
  INSERT INTO iceberg.default.sales VALUES
    (5, 105, DATE '2024-01-03', 800.00, 'east', 'bob'),
    (6, 106, DATE '2024-01-03', 950.50, 'west', 'alice')
""")

In [ ]:
// Verify partition evolution
spark.sql("DESCRIBE EXTENDED iceberg.default.sales").show(truncate=false)
println("✅ Partition evolution successful!")

## Step 2: Z-Ordering for Data Clustering

In [ ]:
// Create table with Z-ordering
spark.sql("""
  CREATE TABLE iceberg.default.transactions (
    transaction_id STRING,
    user_id INT,
    transaction_date TIMESTAMP,
    amount DECIMAL(10,2),
    merchant STRING,
    category STRING
  ) USING iceberg
  PARTITIONED BY (days(transaction_date))
  ORDER BY user_id, transaction_date
  TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet',
    'write.parquet.compression-codec'='gzip'
  )
""")

In [ ]:
// Insert test data
spark.sql("""
  INSERT INTO iceberg.default.transactions VALUES
    ('txn001', 1, TIMESTAMP '2024-01-01 10:00:00', 100.00, 'amazon', 'electronics'),
    ('txn002', 1, TIMESTAMP '2024-01-01 14:00:00', 50.00, 'grocery', 'food'),
    ('txn003', 2, TIMESTAMP '2024-01-01 11:00:00', 200.00, 'amazon', 'electronics'),
    ('txn004', 2, TIMESTAMP '2024-01-02 09:00:00', 75.00, 'gas', 'fuel')
""")

In [ ]:
// Verify data clustering
spark.sql("""
  SELECT * FROM iceberg.default.transactions
  WHERE user_id = 1
  ORDER BY transaction_date
""").show()
println("✅ Z-ordering configured successfully!")

## Step 3: File Compaction and Maintenance

In [ ]:
// Create table that will need compaction
spark.sql("""
  CREATE TABLE iceberg.default.log_events (
    event_id STRING,
    timestamp TIMESTAMP,
    level STRING,
    message STRING,
    service STRING
  ) USING iceberg
  PARTITIONED BY (service, days(timestamp))
  TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet'
  )
""")

In [ ]:
// Insert data in multiple small batches
for (i <- 1 to 10) {
  spark.sql(s"""
    INSERT INTO iceberg.default.log_events VALUES
    ('evt${i}001', TIMESTAMP '2024-01-01 ${i}:00:00', 'INFO', 'Service started', 'api'),
    ('evt${i}002', TIMESTAMP '2024-01-01 ${i}:05:00', 'DEBUG', 'Request received', 'api'),
    ('evt${i}003', TIMESTAMP '2024-01-01 ${i}:10:00', 'INFO', 'Response sent', 'api')
  """)
}

In [ ]:
// Check file count before compaction
val filesBefore = spark.sql("""
  SELECT COUNT(*) as file_count
  FROM iceberg.default.log_events.files
""").collect().head.getLong(0)
println(s"File count before compaction: $filesBefore")

In [ ]:
// Perform compaction using Spark rewrite
spark.sql("""
  CALL iceberg.system.rewrite_data_files(
    'iceberg.default.log_events',
    map(
      'min-input-files', '5',
      'target-size-bytes', str(128 * 1024 * 1024)
    )
  )
""")

In [ ]:
// Check file count after compaction
val filesAfter = spark.sql("""
  SELECT COUNT(*) as file_count
  FROM iceberg.default.log_events.files
""").collect().head.getLong(0)
println(s"File count after compaction: $filesAfter")

assert(filesAfter < filesBefore, "Compaction should reduce file count")
println("✅ File compaction successful!")

## Step 4: Metadata-Only Query Optimization

In [ ]:
// Create large table for metadata optimization demo
spark.sql("""
  CREATE TABLE iceberg.default.sensor_data (
    sensor_id INT,
    reading_time TIMESTAMP,
    temperature DOUBLE,
    humidity DOUBLE,
    pressure DOUBLE,
    location STRING
  ) USING iceberg
  PARTITIONED BY (location, days(reading_time))
  TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet'
  )
""")

In [ ]:
// Insert data with clear partition boundaries
spark.sql("""
  INSERT INTO iceberg.default.sensor_data VALUES
    (1, TIMESTAMP '2024-01-01 00:00:00', 20.5, 45.0, 1013.25, 'warehouse'),
    (2, TIMESTAMP '2024-01-01 01:00:00', 21.0, 46.0, 1013.30, 'warehouse'),
    (3, TIMESTAMP '2024-01-01 02:00:00', 20.8, 45.5, 1013.20, 'warehouse'),
    (4, TIMESTAMP '2024-01-02 00:00:00', 18.5, 50.0, 1012.80, 'office'),
    (5, TIMESTAMP '2024-01-02 01:00:00', 19.0, 49.0, 1012.90, 'office'),
    (6, TIMESTAMP '2024-01-03 00:00:00', 22.0, 40.0, 1014.00, 'warehouse')
""")

In [ ]:
// Query that should use metadata-only filtering
spark.sql("""
  EXPLAIN EXTENDED
  SELECT * FROM iceberg.default.sensor_data
  WHERE location = 'office'
  AND reading_time >= TIMESTAMP '2024-01-02 00:00:00'
  AND reading_time < TIMESTAMP '2024-01-03 00:00:00'
""").show(truncate=false)

In [ ]:
// Run the actual query
spark.sql("""
  SELECT * FROM iceberg.default.sensor_data
  WHERE location = 'office'
  AND reading_time >= TIMESTAMP '2024-01-02 00:00:00'
  AND reading_time < TIMESTAMP '2024-01-03 00:00:00'
""").show()
println("✅ Metadata-only filtering working!")

## Step 5: Complex Schema Migrations

In [ ]:
// Create table with initial schema
spark.sql("""
  CREATE TABLE iceberg.default.customer_profile (
    customer_id INT,
    name STRING,
    email STRING,
    phone STRING
  ) USING iceberg
  TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet'
  )
""")

In [ ]:
// Insert initial data
spark.sql("""
  INSERT INTO iceberg.default.customer_profile VALUES
    (1, 'Alice', 'alice@example.com', '555-0101'),
    (2, 'Bob', 'bob@example.com', '555-0102')
""")

In [ ]:
// Add new column with default value
spark.sql("""
  ALTER TABLE iceberg.default.customer_profile
  ADD COLUMN created_at TIMESTAMP
""")

In [ ]:
// Drop column
spark.sql("""
  ALTER TABLE iceberg.default.customer_profile
  DROP COLUMN phone
""")

In [ ]:
// Rename column
spark.sql("""
  ALTER TABLE iceberg.default.customer_profile
  RENAME COLUMN email TO contact_email
""")

In [ ]:
// Verify schema changes
spark.sql("DESCRIBE iceberg.default.customer_profile").show()
spark.sql("SELECT * FROM iceberg.default.customer_profile").show()
println("✅ Complex schema migrations successful!")

## ✅ Lab Completion Checklist

- [x] Partition evolution successfully implemented
- [x] Z-ordering configured for data clustering
- [x] File compaction reduces file count
- [x] Metadata-only query optimization working
- [x] Complex schema migrations preserve data integrity

## 🎓 Key Concepts Learned

1. **Partition Evolution**: Adding partition fields to existing tables
2. **Z-Ordering**: Data clustering for improved query performance
3. **File Compaction**: Merging small files for better performance
4. **Metadata-Only Filtering**: Skipping files based on partition metadata
5. **Schema Migrations**: Complex schema changes while preserving data